## Problem Statement

### Business Context

Airbnb is an online platform that allows people to rent short-term accommodation. This ranges from regular people with a spare bedroom to property management firms who lease multiple rentals. On the one side, Airbnb enables owners to list their space and earn rental money. On the other side, it provides travelers easy access to renting private homes.

Airbnb receives commissions from two sources upon every booking, namely from the hosts and guests. For every booking, Airbnb charges the guest 6-12% of the booking fee. Moreover, Airbnb charges the host 3% for every successful transaction.

As a senior data scientist at Airbnb, you have to come up with a pricing model that can effectively predict the Rent for accommodation and can help hosts, travelers, and also the business in devising profitable strategies.

### Objective

To explore and visualize the data, build a linear regression model to predict the prices of Airbnb rental rooms, and generate a set of insights and recommendations that will help the business.

### Data Description

The data contains information about the different types of rental rooms offered by Airbnb over a fixed period of time. The detailed data dictionary is given below.

**Data Dictionary**

- id: Property ID
- room_type: Type of Room in the property
- accommodates: How many adults can this property accommodate
- bathrooms: Number of bathrooms on the property
- cancellation_policy: Cancellation policy of the property
- cleaning_fee: This denotes whether the property cleaning fee is included in the rent or not
- instant_bookable: It indicates whether an instant booking facility is available or not
- review_scores_rating: Review rating score of the property
- bedrooms: Number of bedrooms in the property
- beds: Total number of beds in the property
- log_price: Log of the rental price of the property for a fixed period. [If the price is 12000 dollars, then log_price represents log(12000)]

## **Please read the instructions carefully before starting the project.**

This is a commented Python Notebook file in which all the instructions and tasks to be performed are mentioned.

* Blanks '_______' are provided in the notebook that need to be filled with an appropriate code to get the correct result

* With every '_______' blank, there is a comment that briefly describes what needs to be filled in the blank space

* Identify the task to be performed correctly and only then proceed to write the required code

* Fill the code wherever asked by the commented lines like "# write your code here" or "# complete the code"

* Running incomplete code may throw an error

* Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors

* Add the results/observations derived from the analysis in the presentation and submit the same in .pdf format

## Importing necessary libraries

In [ ]:
# Installing the libraries with the specified version.
!pip install numpy==1.25.2 pandas==1.5.3 matplotlib==3.7.1 seaborn==0.13.1 scikit-learn==1.2.2 sklearn-pandas==2.2.0 -q --user

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# Libraries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 200)

# to split the data into train and test
from sklearn.model_selection import train_test_split

# to build linear regression_model
from sklearn.linear_model import LinearRegression

# to check model performance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# to suppress warnings
import warnings

warnings.filterwarnings("ignore")

## Loading the dataset

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# loading the dataset
data = pd.read_csv("______")

## Data Overview

### Displaying the first few rows of the dataset

In [ ]:
data.'_______' ## Complete the code to get the shape of data

### Checking the shape of the dataset

In [ ]:
# checking shape of the data
print(f"There are {data.shape[0]} rows and {data.shape[1]} columns.")

### Checking 10 random rows of the dataset

In [ ]:
# let's view a sample of the data
data.sample(n=10, random_state=1)

In [ ]:
# drop the id column as it does not add any value to the analysis
data.drop("id", axis=1, inplace=True)

In [ ]:
# let's create a copy of the data to avoid any changes to original data
df = data.copy()

### Checking the data types of the columns for the dataset

In [ ]:
# checking column datatypes and number of non-null values
df.info()

### Checking for duplicate values

In [ ]:
df.'_______' ## Complete the code to check duplicate entries in the data

In [ ]:
df.drop_duplicates(inplace=True)
df.shape

### Statistical summary of the dataset

In [ ]:
df.'_______' ## Complete the code to print the statistical summary of the data

**Let's check the unique values for categorical variables.**

In [ ]:
# list of all categorical variables
cat_col = df.select_dtypes(include="object").columns.tolist()

# printing the number of occurrences of each unique value in each categorical column
for column in cat_col:
    print(df[column].value_counts())
    print("-" * 50)

**Let's convert the values of '*instant_bookable*' from '*f*' and '*t*' to boolean.**

In [ ]:
df.instant_bookable.replace(["f", "t"], [False, True], inplace=True)

## <a name='link2'>Exploratory Data Analysis (EDA) Summary</a>


**The below functions need to be defined to carry out the Exploratory Data Analysis.**

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 1, 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n].sort_values(),
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

In [ ]:
# function to plot a boxplot and a histogram along the same scale.


def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to the show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a star will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette="winter"
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

### Univariate Analysis

***log_price* (the target variable)**

In [ ]:
histogram_boxplot(df, "log_price")

**Actual rental price**

In [ ]:
df["act_price"] = np.exp(df["log_price"])
histogram_boxplot('_______')  ## Complete the code to create histogram_boxplot for 'act_price'

In [ ]:
# drop the act_price column
df.drop("act_price", axis=1, inplace=True)

***review_scores_rating***

In [ ]:
histogram_boxplot('_______')  ## Complete the code to create histogram_boxplot for 'review_scores_rating'

***beds***

In [ ]:
histogram_boxplot('_______')  ## Complete the code to create histogram_boxplot for 'beds'

In [ ]:
labeled_barplot(df, "beds", perc=True)

***bedrooms***

In [ ]:
labeled_barplot('_______') ## Complete the code to create labeled_barplot for 'bedrooms'

***bathrooms***

In [ ]:
labeled_barplot('_______') ## Complete the code to create labeled_barplot for 'bathrooms'

***accommodates***

In [ ]:
histogram_boxplot('_______')  ## Complete the code to create histogram_boxplot for 'accommodates'

In [ ]:
labeled_barplot('_______') ## Complete the code to create labeled_barplot for 'accommodates'

***room_type***

In [ ]:
labeled_barplot('_______') ## Complete the code to create labeled_barplot for 'room_type'

***cancellation_policy***

In [ ]:
labeled_barplot('_______') ## Complete the code to create labeled_barplot for 'cancellation_policy'

***cleaning_fee***

In [ ]:
labeled_barplot('_______') ## Complete the code to create labeled_barplot for 'cleaning_fee'

***instant_bookable***

In [ ]:
labeled_barplot('_______') ## Complete the code to create labeled_barplot for 'instant_bookable'

## Bivariate Analysis

**Let's check the correlation between numerical columns.**

In [ ]:
plt.figure(figsize=(15, 7))
sns.heatmap(df.corr(numeric_only = True), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral")
plt.show()

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
sns.pairplot(data=df[numeric_columns], diag_kind="kde")
plt.show()

**Let's check the variation in *log_price* with some of the other variables.**

***accommodates* vs *log price***

In [ ]:
plt.figure(figsize=(15, 7))
sns.boxplot(x="accommodates", y="log_price", data=df)
plt.show()

In [ ]:
plt.figure(figsize=(18, 5))
sns.boxplot('______') ## Complete the code to create a boxplot for 'accommodates' and 'log_price'
plt.show()

***bedrooms* vs *log_price***

In [ ]:
plt.figure(figsize=(18, 5))
sns.boxplot('______') ## Complete the code to create a boxplot for 'bedrooms' and 'log_price'
plt.show()

In [ ]:
plt.figure(figsize=(20, 7))
sns.boxplot(x="bedrooms", y="log_price", data=df, hue="room_type")
plt.show()

## Data Preprocessing

### Missing Value Treatment

In [ ]:
df.isnull().sum()

In [ ]:
# list of columns for which missing values will be dropped
nonnumeric_columns = df.select_dtypes(exclude=np.number).columns
df.dropna(subset=nonnumeric_columns, inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
# list of columns for which missing values will be imputed
numeric_columns.remove("log_price")

df[numeric_columns] = df.groupby(["_____"])[numeric_columns].transform(
    lambda x: x.fillna(x.median())
) # complete the code to impute missing values in numerical columns with median grouped by room_type

In [ ]:
df.'_____'  # check the number of missing values

In [ ]:
df.describe(include="all").T

In [ ]:
# adding small positive value to log_price
eps = 1e-06
df["log_price"] = df["log_price"] + eps

df["log_price"].describe()

In [ ]:
bool_cols = ["cleaning_fee", "instant_bookable"]
df[bool_cols] = df[______].astype(int) # complete the code to conver the boolean columns to integer type

df.head()

### Outlier Detection

In [ ]:
# outlier detection using boxplot
numeric_columns = df.select_dtypes(include=np.number)
plt.figure(figsize=(20, 30))

for i, variable in enumerate(numeric_columns):
    plt.subplot(5, 4, i + 1)
    plt.boxplot(df[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

### Outlier Treatment

- Let's treat all the outliers by flooring and capping.

In [ ]:
def treat_outliers(df, col):
    """
    treats outliers in a varaible
    col: str, name of the numerical varaible
    df: data frame
    col: name of the column
    """
    Q1 = df[col].quantile(0.25)  # 25th quantile
    Q3 = df[col].quantile(____)  # complete the code for 75th quantile
    IQR = Q3 - Q1
    Lower_Whisker = Q1 - 1.5 * IQR
    Upper_Whisker = Q3 + 1.5 * IQR
    df[col] = np.clip(
        df[col], Lower_Whisker, Upper_Whisker
    )  # all the values samller than Lower_Whisker will be assigned value of Lower_whisker
    # and all the values above upper_whishker will be assigned value of upper_Whisker
    return df


def treat_outliers_all(df, col_list):
    """
    treat outlier in all numerical varaibles
    col_list: list of numerical varaibles
    df: data frame
    """
    for c in col_list:
        df = treat_outliers(df, c)

    return df

In [ ]:
# treating the outliers
numerical_col = df.select_dtypes(include=np.number).columns.tolist()
df = treat_outliers_all(df, numerical_col)

In [ ]:
# let's look at the boxplots to see if the outliers have been treated or not
plt.figure(figsize=(20, 30))

for i, variable in enumerate(numeric_columns):
    plt.subplot(5, 4, i + 1)
    plt.boxplot(df[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

## Model Building - Linear Regression


### Model Performance Check

**Let's check the performance of the model using different metrics.**

* We will be using metric functions defined in sklearn for RMSE, MAE, and $R^2$.
* We will define a function to calculate MAPE and adjusted $R^2$.    
* We will create a function which will print out all the above metrics in one go.

In [ ]:
# function to compute adjusted R-squared
def adj_r2_score(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))


# function to compute MAPE
def mape_score(targets, predictions):
    return np.mean(np.abs(targets - predictions) / targets) * 100


# function to compute different metrics to check performance of a regression model
def model_performance_regression(model, predictors, target):
    """
    Function to compute different metrics to check regression model performance

    model: regressor
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    r2 = r2_score(target, pred)  # to compute R-squared
    adjr2 = adj_r2_score(predictors, target, pred)  # to compute adjusted R-squared
    rmse = np.sqrt(mean_squared_error(target, pred))  # to compute RMSE
    mae = mean_absolute_error(target, pred)  # to compute MAE
    mape = mape_score(target, pred)  # to compute MAPE

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {
            "RMSE": rmse,
            "MAE": mae,
            "R-squared": r2,
            "Adj. R-squared": adjr2,
            "MAPE": mape,
        },
        index=[0],
    )

    return df_perf

1. We want to predict the log of rental price.

2. Before we proceed to build a model, we'll have to encode categorical features.

3. We'll split the data into train and test to be able to evaluate the model that we build on the train data.

4. We will build a Linear Regression model using the train data and then check it's performance.

### Data Preparation for modeling with log_price as Dependent variable

In [ ]:
# defining the dependent and independent variables
X = df.drop(["log_price"], axis=1)
y = df["__________"] # complete the code to set 'log_price' as the target variable

In [ ]:
# creating dummy variables
X = pd.'_______'(
    X,
    columns=X.select_dtypes(include=["object", "category"]).columns.tolist(),
    drop_first=True,
)  ## Complete the code to create dummies for independent features

X = X.astype(float)

X.head()

In [ ]:
# splitting the data in 70:30 ratio for train to test data

x_train, x_test, y_train, y_test = '_______' ## Complete the code to split the data into train and test in specified ratio

In [ ]:
print("Number of rows in train data =", x_train.shape[0])
print("Number of rows in test data =", x_test.shape[0])

### Model Building - Linear Regression with log_price as Dependent variable

In [ ]:
# fitting the linear model
lin_reg_model = LinearRegression()
lin_reg_model.fit(x_train, y_train)

In [ ]:
# Checking model performance on train set
lin_reg_model_perf_train = model_performance_regression(lin_reg_model, x_train, y_train)
lin_reg_model_perf_train

In [ ]:
# checking model performance on test set
print("Test Performance\n")
lin_reg_model_perf_test = model_performance_regression('_______') ## Complete the code to check the performance on test data
lin_reg_model_perf_test

## Business Insights and Recommendations

-


___